# Colombian Government Contracts and Investment Projects Analysis

This analysis examines procurement data from Colombia's public contracting system (SECOP) and the national project bank (BPIN). The notebook demonstrates scalable data processing techniques using Apache Spark to analyze government spending patterns, supplier concentration, and procurement trends across multiple years.

**Data Sources:**
- SECOP (Sistema Electrónico para la Contratación Pública): Contract registry from Colombian government procurement
- BPIN (Banco de Proyectos de Inversión Nacional): Investment project database

## Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

## Data Loading

The SECOP data is stored in JSON format with complex nested structures. The loading strategy employs sampling-based schema inference to optimize performance on large datasets. BPIN data in CSV format is loaded with automatic type detection.

In [0]:
secop_path = "wasbs://sid@uniandesyjt.blob.core.windows.net/secop"
bpin_path = "wasbs://sid@uniandesyjt.blob.core.windows.net/bpin"

# Load SECOP contract data
df_secop = spark.read \
    .format("json") \
    .option("multiline", "true") \
    .option("samplingRatio", "0.1") \
    .load(secop_path)

# Load BPIN investment project data
df_bpin = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bpin_path)

## Top Suppliers by Contract Value (2024)

Analysis of the leading suppliers by total contract value in 2024, identifying the key vendors in government procurement.

In [0]:
top_suppliers = df_secop \
    .filter(F.col("anno_firma") == 2024) \
    .groupBy("proveedor_adjudicado") \
    .agg(F.sum("valor_del_contrato").alias("total_value")) \
    .orderBy(F.desc("total_value")) \
    .limit(10)

print("TOP 10 SUPPLIERS BY CONTRACT VALUE - 2024\n")
results = top_suppliers.collect()
for idx, row in enumerate(results, 1):
    print(f"{idx:2}. {row['proveedor_adjudicado']:<50} ${row['total_value']:>15,}")

TOP 10 SUPPLIERS BY CONTRACT VALUE - 2024

 1. MARIA SORELY GRISALES / CREACIONES BJ                $2,115,156,861,243
 2. UNIVERSIDAD DE ANTIOQUIA                             $1,856,417,293,206
 3. INSTITUTO PARA EL DESARROLLO DE ANTIOQUIA IDEA       $1,703,797,668,980
 4. FINANCIERA DE DESARROLLO TERRITORIAL S.A.            $1,541,604,758,133
 5. INSTITUTO TECNOLOGICO METROPOLITANO                  $1,152,177,220,033
 6. BANCOLOMBIA                                          $1,022,475,569,903
 7. SOCIEDAD DE ACTIVOS ESPECIALES SAS                   $1,000,549,529,167
 8. ESU                                                    $870,387,131,033
 9. correagro                                              $841,630,245,974
10. UNIDAD PARA LAS VÍCTIMAS - FONDO PARA LA REPARACIÓ     $710,000,000,000


## Contract Value Distribution by Year

Summary statistics of contract values across different years, including average and median values to understand spending patterns.

In [0]:
value_stats = df_secop \
    .groupBy("anno_firma") \
    .agg(
        F.count("valor_del_contrato").alias("contract_count"),
        F.avg("valor_del_contrato").alias("average_value"),
        F.expr("approx_percentile(valor_del_contrato, 0.5)").alias("median_value"),
        F.min("valor_del_contrato").alias("min_value"),
        F.max("valor_del_contrato").alias("max_value")
    ) \
    .orderBy(F.desc("anno_firma"))

print("CONTRACT VALUE STATISTICS BY YEAR")
value_stats.show()

CONTRACT VALUE STATISTICS BY YEAR


## Supplier Market Concentration

Evaluation of market concentration by identifying suppliers with the highest contract volumes, indicating potential procurement concentration risks or market consolidation.

In [0]:
supplier_concentration = df_secop \
    .groupBy("proveedor_adjudicado") \
    .agg(F.count("*").alias("contract_count")) \
    .orderBy(F.desc("contract_count")) \
    .limit(20) \
    .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("contract_count"))))

print("TOP 20 SUPPLIERS BY CONTRACT VOLUME")
supplier_concentration.select(
    "rank",
    F.col("proveedor_adjudicado").alias("supplier"),
    "contract_count"
).show()

TOP 20 SUPPLIERS BY CONTRACT VOLUME


## Market Entry Analysis: New Suppliers in 2024

Identification of new suppliers entering the procurement market in 2024 with no prior contracts in 2020. This analysis reveals supplier base changes and market dynamics.

In [0]:
suppliers_2024 = df_secop \
    .filter(F.col("anno_firma") == 2024) \
    .select("proveedor_adjudicado").distinct()

suppliers_2020 = df_secop \
    .filter(F.col("anno_firma") == 2020) \
    .select("proveedor_adjudicado").distinct()

# Identify suppliers present in 2024 but not in 2020
new_suppliers = suppliers_2024 \
    .join(suppliers_2020, "proveedor_adjudicado", "left_anti")

new_count = new_suppliers.count()
print("NEW SUPPLIERS ENTERING MARKET IN 2024")
print(f"Suppliers with 2024 contracts but no history in 2020: {new_count:,}")
print()
new_suppliers.show(20, truncate=False)

NEW SUPPLIERS ENTERING MARKET IN 2024
Suppliers with 2024 contracts but no history in 2020: 125,432


## Service Provision Contracts Analysis

Statistical overview of service provision contracts, which represent a significant portion of government procurement activities. Analysis includes distribution metrics.

In [0]:
service_contracts = df_secop \
    .filter(F.col("objeto_del_contrato").contains("Prestación de servicios"))

service_stats = service_contracts.agg(
    F.count("valor_del_contrato").alias("total_contracts"),
    F.avg("valor_del_contrato").alias("average_value"),
    F.expr("approx_percentile(valor_del_contrato, 0.5)").alias("median_value"),
    F.sum("valor_del_contrato").alias("total_volume")
)

print("SERVICE PROVISION CONTRACTS - SUMMARY STATISTICS")
service_stats.show()

SERVICE PROVISION CONTRACTS - SUMMARY STATISTICS


## Supplier Diversity Over Time

Tracking the number of unique suppliers in each year, indicating trends in supplier base growth and market diversification.

In [0]:
suppliers_by_year = df_secop \
    .groupBy("anno_firma") \
    .agg(F.approx_count_distinct("proveedor_adjudicado").alias("unique_suppliers")) \
    .orderBy(F.desc("anno_firma")) \
    .withColumnRenamed("anno_firma", "year")

print("UNIQUE SUPPLIERS BY YEAR")
suppliers_by_year.show()

UNIQUE SUPPLIERS BY YEAR


## Procurement Focus Areas: Text Analysis

Word frequency analysis of contract descriptions reveals primary focus areas and priorities in government procurement. This unstructured data analysis identifies dominant procurement categories and government service delivery priorities.

In [0]:
frequent_terms = df_secop \
    .select(F.explode(F.split(F.lower(F.col("objeto_del_contrato")), " ")).alias("term")) \
    .filter(F.length(F.col("term")) > 3) \
    .groupBy("term") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("count"))))

print("TOP 20 TERMS IN CONTRACT DESCRIPTIONS")
frequent_terms.select("rank", "term", "count").show(truncate=False)

TOP 20 TERMS IN CONTRACT DESCRIPTIONS
